> 📂 **Practice Set 3: Hashmaps, Trees and Heaps** · [assignments](../README.md) · [course home](../../README.md)

# Practice Set 3: Hashmaps, Trees and Heaps (Solutions)

**Covers:** Handbook chapters 18 to 21

Dictionaries for lookup, recursion on trees, and heaps for top-k. A `TreeNode` class and a `build(values)` helper (level-order, `None` for a gap) are provided.

Compare with your own attempt. Try each problem yourself first!

## Setup

Run this cell once. It prepares what the problems and their checks need (for example the `check_recursive` helper or a `Node` class).

In [1]:
from collections import deque


class TreeNode:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right


def build(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue, i = deque([root]), 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root


import inspect
import re


def check_recursive(fn, allow_loops=False):
    """Passes when `fn` calls itself and, unless allow_loops=True, uses no for/while statements."""
    lines = inspect.getsource(fn).split("\n")[1:]
    body = "\n".join(lines)
    assert re.search(rf"\b{fn.__name__}\s*\(", body), f"{fn.__name__} should call itself (recursion)"
    if not allow_loops:
        assert not re.search(r"^\s*(for|while)\b", body, re.M), "No loops here: use recursion instead"


## Problem 1: Two sum

Write `two_sum(numbers, target)` returning the indices `(i, j)` with `i < j` of two numbers that add to `target`, or `None`. Use one pass and a dictionary.

In [2]:
def two_sum(numbers, target):
    seen = {}
    for i, value in enumerate(numbers):
        if target - value in seen:
            return seen[target - value], i
        seen[value] = i
    return None

In [3]:
assert two_sum([2, 7, 11, 15], 9) == (0, 1)
assert two_sum([3, 2, 4], 6) == (1, 2)
assert two_sum([1, 2], 10) is None
print('✅ Problem 1 passed')

✅ Problem 1 passed


## Problem 2: Group the anagrams

Write `group_anagrams(words)` returning a list of groups (lists) of words that are anagrams of each other. Group and word order do not matter.

In [4]:
def group_anagrams(words):
    groups = {}
    for word in words:
        groups.setdefault(''.join(sorted(word)), []).append(word)
    return list(groups.values())

In [5]:
result = group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat'])
assert sorted(sorted(g) for g in result) == [['ate', 'eat', 'tea'], ['bat'], ['nat', 'tan']]
assert group_anagrams([]) == []
print('✅ Problem 2 passed')

✅ Problem 2 passed


## Problem 3: Maximum depth

Write `max_depth(root)`: the number of nodes on the longest path from the root down to a leaf (0 for an empty tree).

In [6]:
def max_depth(root):
    if root is None:
        return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

In [7]:
assert max_depth(None) == 0 and max_depth(TreeNode(1)) == 1
assert max_depth(build([1, 2, 3, 4, 5, None, 6])) == 3
assert max_depth(build([1, 2, None, 3])) == 3
check_recursive(max_depth)
print('✅ Problem 3 passed')

✅ Problem 3 passed


## Problem 4: Validate a binary search tree

Write `is_valid_bst(root)`. Remember: every node must lie within the bounds set by *all* its ancestors, not just its parent.

In [8]:
def is_valid_bst(root, low=float('-inf'), high=float('inf')):
    if root is None:
        return True
    if not (low < root.value < high):
        return False
    return is_valid_bst(root.left, low, root.value) and is_valid_bst(root.right, root.value, high)

In [9]:
assert is_valid_bst(build([2, 1, 3])) and is_valid_bst(None)
assert not is_valid_bst(build([5, 1, 4, None, None, 3, 6]))
assert not is_valid_bst(build([5, 3, 8, 1, 6]))      # 6 is bigger than its parent 3 but sits left of 5
print('✅ Problem 4 passed')

✅ Problem 4 passed


## Problem 5: Lowest common ancestor

Write `lowest_common_ancestor(root, p, q)` returning the **value** of the deepest node that has both values `p` and `q` in its subtree (a node counts as its own ancestor).

In [10]:
def lowest_common_ancestor(root, p, q):
    def find(node):
        if node is None or node.value in (p, q):
            return node
        left, right = find(node.left), find(node.right)
        if left and right:
            return node
        return left or right

    found = find(root)
    return found.value if found else None

In [11]:
tree = build([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
assert lowest_common_ancestor(tree, 5, 1) == 3
assert lowest_common_ancestor(tree, 5, 4) == 5
assert lowest_common_ancestor(tree, 6, 4) == 5
assert lowest_common_ancestor(tree, 7, 8) == 3
print('✅ Problem 5 passed')

✅ Problem 5 passed


## Problem 6: The k largest numbers

Write `k_largest(numbers, k)` returning the `k` largest numbers, biggest first. Use a heap of size `k` (chapter 21).

In [12]:
import heapq


def k_largest(numbers, k):
    heap = []
    for x in numbers:
        if len(heap) < k:
            heapq.heappush(heap, x)
        elif x > heap[0]:
            heapq.heapreplace(heap, x)
    return sorted(heap, reverse=True)

In [13]:
assert k_largest([4, 9, 1, 7, 3, 8, 2, 10, 5], 3) == [10, 9, 8]
assert k_largest([1], 3) == [1] and k_largest([], 2) == []
print('✅ Problem 6 passed')

✅ Problem 6 passed


## Problem 7: Level order

Write `level_order(root)` returning a list of lists: the values of each level, from the top, left to right.

In [14]:
def level_order(root):
    levels = []
    queue = deque([root] if root else [])
    while queue:
        level = []
        for _ in range(len(queue)):
            node = queue.popleft()
            level.append(node.value)
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
        levels.append(level)
    return levels

In [15]:
assert level_order(build([1, 2, 3, 4, 5, None, 6])) == [[1], [2, 3], [4, 5, 6]]
assert level_order(None) == [] and level_order(TreeNode(9)) == [[9]]
print('✅ Problem 7 passed')

✅ Problem 7 passed
